# Visual Regression Edge Cases

Example charts that exercise common ggplot2 workflows and high-risk rendering surfaces.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

repo = Path.cwd()
if not (repo / 'ggplotly').exists():
    repo = next(parent for parent in Path.cwd().parents if (parent / 'ggplotly').exists())
sys.path.insert(0, str(repo))

from ggplotly import *

rng = np.random.default_rng(42)

## Cleveland Dot Plot

In [ ]:
dot = pd.DataFrame({
    'category': ['Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon'],
    'value': [42, 57, 35, 64, 49]
}).sort_values('value')

(ggplot(dot, aes(x='value', y='category'))
 + geom_point(size=12, color='steelblue'))

## Dumbbell Chart

In [ ]:
dumbbell = pd.DataFrame({
    'item': ['A', 'B', 'C', 'D', 'E'],
    'before': [34, 45, 29, 51, 40],
    'after': [48, 52, 41, 59, 44]
})

(ggplot(dumbbell, aes(y='item'))
 + geom_segment(aes(x='before', xend='after', yend='item'), color='lightgray', size=4)
 + geom_point(aes(x='before'), color='#1f77b4', size=10)
 + geom_point(aes(x='after'), color='#d62728', size=10)
 + labs(x='value', y='item'))

## Slopegraph

In [ ]:
slope = pd.DataFrame({
    'stage': ['1 Before', '2 After'] * 5,
    'item': np.repeat(['A', 'B', 'C', 'D', 'E'], 2),
    'value': [34, 48, 45, 52, 29, 41, 51, 59, 40, 44]
})

(ggplot(slope, aes(x='stage', y='value', group='item', color='item'))
 + geom_line(size=3)
 + geom_point(size=8)
 + scale_color_viridis_d())

## Raincloud Plot

In [ ]:
rain = pd.DataFrame({
    'group': ['control'] * 90 + ['treatment'] * 90,
    'value': np.r_[rng.normal(0, 0.8, 90), rng.normal(1.1, 0.65, 90)]
})

(ggplot(rain, aes(x='group', y='value'))
 + geom_violin(fill='rgba(31,119,180,0.22)', color='steelblue')
 + geom_boxplot(width=0.22, fill='white', color='black')
 + geom_jitter(width=0.18, alpha=0.45, size=5, color='darkslategray'))

## Ridgeline Density Plot

In [ ]:
ridge = pd.DataFrame({
    'cohort': np.repeat(['Q1', 'Q2', 'Q3', 'Q4'], 100),
    'score': np.r_[rng.normal(0, .6, 100), rng.normal(.5, .7, 100), rng.normal(1, .5, 100), rng.normal(1.4, .8, 100)]
})

(ggplot(ridge, aes(x='score', y='cohort'))
 + geom_density_ridges(scale=0.75, fill='rgba(44,160,44,0.32)', color='#2ca02c'))

## Calendar Heatmap

In [ ]:
dates = pd.date_range('2026-01-01', periods=84, freq='D')
calendar = pd.DataFrame({
    'week': ((dates.dayofyear - 1) // 7) + 1,
    'weekday': dates.weekday,
    'value': 10 + np.sin(np.arange(len(dates)) / 5) * 4 + rng.normal(0, 1, len(dates))
})

(ggplot(calendar, aes(x='week', y='weekday', fill='value'))
 + geom_tile(palette='Viridis')
 + scale_y_continuous(breaks=[0, 1, 2, 3, 4, 5, 6], labels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']))

## Missingness Heatmap

In [ ]:
rows = range(1, 31)
missing = pd.DataFrame([
    {'row': row, 'variable': var, 'missing': int((row + i * 3) % (i + 4) == 0)}
    for row in rows
    for i, var in enumerate(['age', 'income', 'region', 'score', 'status'])
])

(ggplot(missing, aes(x='row', y='variable', fill='missing'))
 + geom_tile(palette='Blues'))

## Annotated Outlier Plot

In [ ]:
out = pd.DataFrame({'x': rng.normal(size=80), 'y': rng.normal(size=80)})
out.loc[[7, 42, 63], ['x', 'y']] = [[2.5, 2.8], [-2.6, 2.4], [2.8, -2.2]]
out['label'] = [''] * len(out)
out.loc[[7, 42, 63], 'label'] = ['high-high', 'low-high', 'high-low']
outliers = out[out['label'] != '']

(ggplot(out, aes(x='x', y='y'))
 + geom_point(alpha=0.55, color='gray')
 + geom_point(outliers, aes(x='x', y='y'), color='crimson', size=10)
 + geom_text_repel(outliers, aes(x='x', y='y', label='label'), color='crimson', force=0.22))

## Faceted Choropleth Map

In [ ]:
states = pd.DataFrame({
    'state': ['CA', 'TX', 'NY', 'FL', 'IL', 'WA'] * 2,
    'value': [100, 84, 76, 69, 58, 63, 92, 88, 81, 72, 61, 70],
    'year': ['2025'] * 6 + ['2026'] * 6
})

(ggplot(states, aes(map_id='state', fill='value'))
 + geom_map(map_type='usa', palette='Viridis')
 + facet_wrap('year'))

## Inset / Multi-Panel Map Example

In [ ]:
map_fig = (ggplot(states[states['year'] == '2026'], aes(map_id='state', fill='value')) + geom_map(map_type='usa')).draw()
rank_fig = (ggplot(states[states['year'] == '2026'].sort_values('value'), aes(x='state', y='value')) + geom_col(fill='steelblue')).draw()

combo = make_subplots(rows=1, cols=2, specs=[[{'type': 'geo'}, {'type': 'xy'}]], subplot_titles=['Map', 'Ranked values'])
for trace in map_fig.data:
    combo.add_trace(trace, row=1, col=1)
for trace in rank_fig.data:
    combo.add_trace(trace, row=1, col=2)
combo.update_geos(scope='usa')
combo.update_layout(height=420, showlegend=False)
HTML(combo.to_html(full_html=False, include_plotlyjs='cdn'))

## Paired Points / Connected Strip Plot

In [ ]:
paired = pd.DataFrame({
    'id': np.repeat([f'S{i}' for i in range(1, 16)], 2),
    'time': ['before', 'after'] * 15,
    'value': np.ravel(np.column_stack([rng.normal(5, 1, 15), rng.normal(6.1, 1, 15)]))
})

(ggplot(paired, aes(x='time', y='value', group='id'))
 + geom_line(alpha=0.35, color='gray')
 + geom_point(size=8, color='steelblue'))

## Log-Scale Histogram

In [ ]:
log_data = pd.DataFrame({'value': rng.lognormal(mean=1.2, sigma=0.75, size=500)})
(ggplot(log_data, aes(x='value')) + geom_histogram(bins=35, fill='seagreen') + scale_y_log10())

## Stacked Area With Negative Values

In [ ]:
x = np.arange(12)
area = pd.DataFrame({
    'x': np.tile(x, 3),
    'series': np.repeat(['positive', 'mixed', 'negative'], len(x)),
    'value': np.r_[np.sin(x / 2) + 2, np.cos(x / 2) - 0.2, -np.sin(x / 3) - 0.7]
})

(ggplot(area, aes(x='x', y='value', fill='series'))
 + geom_area(position='stack', alpha=0.55))

## Dual Encoding Bubble Map

In [ ]:
cities = pd.DataFrame({
    'city': ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Seattle'],
    'lat': [40.7128, 34.0522, 41.8781, 29.7604, 47.6062],
    'lon': [-74.0060, -118.2437, -87.6298, -95.3698, -122.3321],
    'population': [8.3, 3.9, 2.7, 2.3, 0.75],
    'growth': [1.2, 0.8, -0.2, 1.5, 2.0]
})

(ggplot(cities, aes(x='lon', y='lat', size='population', color='growth', label='city'))
 + geom_map(map_type='usa')
 + geom_point(alpha=0.8))

## Regression Diagnostics Grid

In [ ]:
x = np.linspace(0, 10, 80)
y = 2 + 0.7 * x + rng.normal(0, 0.8, len(x))
fit = np.poly1d(np.polyfit(x, y, 1))(x)
resid = y - fit
diag = pd.concat([
    pd.DataFrame({'x': fit, 'y': resid, 'panel': 'fitted vs residuals'}),
    pd.DataFrame({'x': np.sort(rng.normal(size=len(x))), 'y': np.sort(resid), 'panel': 'qq residuals'}),
    pd.DataFrame({'x': x, 'y': resid, 'panel': 'residuals by x'}),
    pd.DataFrame({'x': np.arange(len(x)), 'y': np.abs(resid), 'panel': 'absolute residuals'})
])

(ggplot(diag, aes(x='x', y='y'))
 + geom_point(alpha=0.6, color='navy')
 + geom_smooth(method='lm', color='crimson')
 + facet_wrap('panel', ncol=2, scales='free'))

## Legend Stress Test

In [ ]:
groups = [f'Very long label {i}' for i in range(1, 9)]
legend_df = pd.DataFrame({
    'x': rng.normal(size=160),
    'y': rng.normal(size=160),
    'group': np.repeat(groups, 20),
    'shape': np.tile(['s1', 's2'], 80),
    'size': np.tile(np.linspace(5, 25, 20), 8)
})

(ggplot(legend_df, aes(x='x', y='y', color='group', shape='shape', size='size'))
 + geom_point(alpha=0.75)
 + scale_color_viridis_d())

## Theme Comparison Matrix

In [ ]:
theme_df = pd.DataFrame({'x': range(1, 8), 'y': [2, 4, 3, 5, 4, 6, 5]})
for name, theme_fn in [('default', theme_default), ('minimal', theme_minimal), ('classic', theme_classic), ('dark', theme_dark)]:
    display(ggplot(theme_df, aes(x='x', y='y')) + geom_line() + geom_point(size=8) + ggtitle(name) + theme_fn())

## Coord Flip With Text Labels

In [ ]:
flip = pd.DataFrame({'category': ['Alpha', 'Beta', 'Gamma', 'Delta'], 'value': [18, 24, 12, 30], 'label': ['18', '24', '12', '30']})

(ggplot(flip, aes(x='category', y='value'))
 + geom_col(fill='steelblue')
 + geom_text(aes(label='label'), vjust=0.5, hjust=1.1, color='white')
 + coord_flip())

## Nested Facet-Like Example

In [ ]:
nested = pd.DataFrame([
    {'x': i, 'y': np.sin(i / 2) + row_i + col_i, 'row': row, 'col': col}
    for row_i, row in enumerate(['North region with long label', 'South region with long label'])
    for col_i, col in enumerate(['Baseline scenario label', 'Optimized scenario label', 'Stress scenario label'])
    for i in range(8)
])

def wrap_nested_label(variable, value):
    return str(value).replace(' with long label', '<br>with long label').replace(' scenario label', '<br>scenario label')

(ggplot(nested, aes(x='x', y='y'))
 + geom_line(color='steelblue')
 + geom_point(size=5)
 + facet_grid('row', 'col', labeller=wrap_nested_label)
 + ggsize(1100, 650))

## Small Multiples With Free Scales

In [ ]:
free = pd.DataFrame([
    {'x': i, 'y': scale * (np.sin(i / 2) + offset), 'panel': panel}
    for panel, scale, offset in [('small', 1, 1), ('medium', 10, 2), ('large', 100, 3)]
    for i in range(12)
])

(ggplot(free, aes(x='x', y='y'))
 + geom_line(color='darkorange')
 + geom_point(size=5)
 + facet_wrap('panel', scales='free'))